# 02 · Outils et boucle agentique : notre premier agent

**Atelier Agents IA · Notebook 2 sur 4**

| | |
|---|---|
| ⏱️ Durée conseillée | 1 heure 10 |
| 🎓 Séquence du cours | Séquences 3 (Outils et boucle) et 4 (Brancher le monde réel) |
| 🧰 Prérequis | Fichier `.env` rempli (voir le README) |

### Ce que vous saurez faire à la fin
- chercher de vrais commerces sur Google Maps avec Apify ;
- décrire un outil en JSON Schema pour que le modèle puisse l'utiliser ;
- écrire la boucle agentique complète, sans framework ;
- gérer les erreurs d'outils ;
- (bonus) donner des documents à l'agent avec le RAG et partager un outil avec MCP.

> **Comment utiliser ce notebook :** exécutez les cellules **une par une, dans l'ordre**, avec `Maj + Entrée`.
> Lisez le texte avant chaque cellule de code : il explique ce qui va se passer.
> Les cellules marquées ✍️ **À vous** sont des exercices ; la solution est cachée juste en dessous (cliquez sur « Voir la solution »).

## 0. Configuration

Le fichier `prospection.py` (dans ce dossier) contient tout le code de ce notebook, bien rangé. Ici on le reconstruit **pas à pas** pour comprendre ; les notebooks 03 et 04 l'importeront directement.

On commence par charger la configuration et vérifier le fichier `.env`.

In [15]:
import os, json
from dotenv import load_dotenv, find_dotenv
import anthropic

load_dotenv(find_dotenv(usecwd=True))
client = anthropic.Anthropic()
MODELE = os.getenv("MODELE", "claude-sonnet-5-5")
MODELE_RAPIDE = os.getenv("MODELE_RAPIDE") or MODELE

from prospection import verifier_configuration
verifier_configuration()

ANTHROPIC_API_KEY : ✅ présente (sk-ant…)
APIFY_TOKEN       : ✅ présente (apify_…) 
Modèle principal  : claude-sonnet-5-5
Modèle rapide     : claude-haiku-5-5


> Si `APIFY_TOKEN` est absent, pas de panique : le notebook fonctionne avec des **commerces fictifs de démonstration**. Ajoutez le jeton dans `.env` quand vous voulez de vraies données.

## 1. Notre premier outil : chercher des commerces avec Apify

[Apify](https://console.apify.com/) propose des « acteurs », des robots prêts à l'emploi. L'acteur **Google Maps Scraper** (`compass/crawler-google-places`) renvoie les commerces que Google Maps affiche pour une recherche.

### 1.1 Récupérer votre jeton Apify
1. Connectez-vous sur [console.apify.com](https://console.apify.com/).
2. Allez dans **Settings → API & Integrations** et copiez votre **Personal API token**.
3. Collez-le dans `.env` après `APIFY_TOKEN=`, puis redémarrez le noyau.

> 💰 **Coût :** l'acteur est payant au résultat (le compte gratuit inclut un crédit mensuel). On limite le nombre de résultats et on **garde les résultats en cache** pour ne jamais payer deux fois la même recherche.

### 1.2 Un appel Apify, tout simplement
Voici l'appel brut, pour voir à quoi ressemblent les données. Il prend 20 à 60 secondes.

In [16]:
jeton = os.getenv("APIFY_TOKEN", "").strip()

if jeton:
    from apify_client import ApifyClient
    apify = ApifyClient(jeton)
    run = apify.actor("compass/crawler-google-places").call(run_input={
        "searchStringsArray": ["boulangerie"],     # ce qu'on taperait dans Google Maps
        "locationQuery": "Dakar Plateau, Sénégal",  # où chercher
        "maxCrawledPlacesPerSearch": 3,             # on limite pour payer peu
        "language": "fr",
        "maxImages": 0,
        "maxReviews": 0,
    })
    id_dataset = run["defaultDatasetId"] if isinstance(run, dict) else run.default_dataset_id
    items = list(apify.dataset(id_dataset).iterate_items())
    print(f"{len(items)} résultats. Premier résultat brut (extrait) :")
    premier = items[0]
    for champ in ["title", "categoryName", "address", "phone", "website", "totalScore", "reviewsCount"]:
        print(f"  {champ:13} : {premier.get(champ)}")
else:
    print("Pas de jeton Apify : on passe directement à la version avec données de démo.")

[apify.crawler-google-places runId:OsuxMTrrkRLb1k2YI] -> Status: RUNNING, Message: 
[apify.crawler-google-places runId:OsuxMTrrkRLb1k2YI] -> 2026-10-30T10:40:11.045Z ACTOR: Pulling container image of build iGk3vkI1QM1iLooFy from registry.
[apify.crawler-google-places runId:OsuxMTrrkRLb1k2YI] -> 2026-10-30T10:40:11.048Z ACTOR: Creating container.
[apify.crawler-google-places runId:OsuxMTrrkRLb1k2YI] -> 2026-10-30T10:40:13.623Z ACTOR: Starting container.
[apify.crawler-google-places runId:OsuxMTrrkRLb1k2YI] -> 2026-10-30T10:40:13.624Z ACTOR: Running under "LIMITED_PERMISSIONS".
[apify.crawler-google-places runId:OsuxMTrrkRLb1k2YI] -> 2026-10-30T10:40:15.576Z INFO  System info {"apifyVersion":"3.7.2","apifyClientVersion":"2.23.4","crawleeVersion":"3.18.0","osType":"Linux","nodeVersion":"v24.22.1"}
[apify.crawler-google-places runId:OsuxMTrrkRLb1k2YI] -> 2026-10-30T10:40:15.973Z INFO  📡 Geolocation started
[apify.crawler-google-places runId:OsuxMTrrkRLb1k2YI] -> 2026-10-30T10:40:16.006Z IN

3 résultats. Premier résultat brut (extrait) :
  title         : Praline
  categoryName  : Pâtisserie
  address       : 14-18, Boulevard El Hadji Djily Mbaye, Dakar 99000, Sénégal
  phone         : +221 33 864 66 74
  website       : None
  totalScore    : 4.5
  reviewsCount  : 69


### 1.3 La fonction `chercher_commerces`
Le résultat brut contient des dizaines de champs. Notre fonction (dans `prospection.py`) :
- garde seulement les champs utiles (nom, adresse, téléphone, note…) ;
- utilise les **données de démo** si aucun jeton n'est configuré ;
- garde les résultats dans `donnees/cache_apify.json` pour ne pas repayer.

In [17]:
from prospection import chercher_commerces

resultats = chercher_commerces("boulangerie", "Dakar Plateau", 3)

import pandas as pd
pd.DataFrame(resultats)

,nom,categorie,adresse,telephone,site_web,note,nb_avis,lien_maps,source
0,Praline,Pâtisserie,"14-18, Boulevard El Hadji Djily Mbaye, Dakar 9...",+221 33 864 66 74,None,4.5,69.0,https://www.google.com/maps/search/?api=1&quer...,Google Maps via Apify
1,"LA BAGUETTE MAGIQUE, 13, RUE ROBERT BRUN PLATE...",Boulangerie,"22 Rue Robert Brun, Sénégal",+221 70 313 68 31,None,4.0,2.0,https://www.google.com/maps/search/?api=1&quer...,Google Maps via Apify
2,Boulangerie Principale,Boulangerie,"Dakar, Sénégal",,None,NaN,NaN,https://www.google.com/maps/search/?api=1&quer...,Google Maps via Apify


💡 **Une idée importante :** un outil, **c'est juste une fonction Python**. Base de données, API de paiement, envoi de SMS : si vous savez l'écrire en Python, votre agent peut l'utiliser.

## 2. Décrire l'outil au modèle (JSON Schema)

Le modèle ne voit pas votre code Python. Il voit seulement une **description** de l'outil : son nom, à quoi il sert, et ses paramètres au format JSON Schema.

> ⚠️ **La description, c'est un prompt.** C'est ce texte que le modèle lit pour décider **quand** et **comment** utiliser l'outil. Une description vague donne un agent qui se trompe.

In [18]:
OUTIL_RECHERCHE = {
    "name": "rechercher_commerces",
    "description": ("Recherche des commerces sur Google Maps au Sénégal par activité et zone. "
                    "Renvoie une liste avec nom, adresse, téléphone et note. "
                    "Une liste vide signifie qu'aucun commerce n'a été trouvé."),
    "input_schema": {
        "type": "object",
        "properties": {
            "activite": {"type": "string", "description": "Type de commerce, ex. boulangerie"},
            "zone":     {"type": "string", "description": "Quartier ou ville, ex. Dakar Plateau"},
            "nombre":   {"type": "integer", "minimum": 1, "maximum": 20,
                         "description": "Nombre de résultats voulus (5 par défaut)"},
        },
        "required": ["activite", "zone"],
    },
}

## 3. Le function calling : le modèle **demande**, votre code **exécute**

On envoie une demande avec la liste des outils disponibles. Regardez bien la réponse : le modèle n'exécute rien, il **écrit une demande d'appel** (un bloc `tool_use`).

In [19]:
rep = client.messages.create(
    model=MODELE, max_tokens=500,
    tools=[OUTIL_RECHERCHE],
    messages=[{"role": "user", "content": "Trouve 3 boulangeries à Dakar Plateau."}],
)

print("stop_reason :", rep.stop_reason)       # 'tool_use' = le modèle veut un outil
for bloc in rep.content:
    print("-", bloc.type, ":", bloc)

stop_reason : tool_use
- tool_use : ToolUseBlock(id='toolu_015hHtkS5VSCuMu7WXRmCkgG', input={'activite': 'boulangerie', 'zone': 'Dakar Plateau', 'nombre': 3}, name='rechercher_commerces', type='tool_use', caller={'type': 'direct'})


Le bloc `tool_use` contient :
- `name` : l'outil voulu ;
- `input` : les paramètres choisis par le modèle ;
- `id` : un identifiant, qu'il faudra **rappeler** en renvoyant le résultat.

**C'est maintenant à notre code d'exécuter l'outil**, puis de renvoyer le résultat au modèle.

In [20]:
demande_outil = next(b for b in rep.content if b.type == "tool_use")
resultat = chercher_commerces(**demande_outil.input)      # notre code exécute l'outil

suite = client.messages.create(
    model=MODELE, max_tokens=800, tools=[OUTIL_RECHERCHE],
    messages=[
        {"role": "user", "content": "Trouve 3 boulangeries à Dakar Plateau."},
        {"role": "assistant", "content": rep.content},                       # sa demande
        {"role": "user", "content": [{"type": "tool_result",                 # notre réponse
                                      "tool_use_id": demande_outil.id,
                                      "content": json.dumps(resultat, ensure_ascii=False)}]},
    ],
)
print("".join(b.text for b in suite.content if b.type == "text"))

Voici 3 boulangeries à Dakar Plateau :

1. **Praline** (pâtisserie)
   - Adresse : 14-18, Boulevard El Hadji Djily Mbaye
   - Téléphone : +221 33 864 66 74
   - Note : 4,5/5 (69 avis)
   - [Voir sur Maps](https://www.google.com/maps/search/?api=1&query=Praline&query_place_id=ChIJqXvNI5ZzwQ4RNC4N26tfSGo)

2. **La Baguette Magique** (boulangerie)
   - Adresse : 22 Rue Robert Brun (l'établissement indique aussi le 13, rue Robert Brun)
   - Téléphone : +221 70 313 68 31
   - Note : 4/5, mais seulement 2 avis
   - [Voir sur Maps](https://www.google.com/maps/search/?api=1&query=LA%20BAGUETTE%20MAGIQUE%2C%2013%2C%20RUE%20ROBERT%20BRUN%20PLATEAU%20DAKAR-PLATEAU%2C%20DAKAR%20SENEGAL&query_place_id=ChIJ22ndw6VzwQ4Rel4W9O4wWuU)

3. **Boulangerie Principale** (boulangerie)
   - Adresse : Dakar, Sénégal (adresse peu précise)
   - Téléphone et note : non renseignés
   - [Voir sur Maps](https://www.google.com/maps/search/?api=1&query=Boulangerie%20Principale&query_place_id=ChIJydDfhVFywQ4RFpqU9KmXAhI

## 4. La boucle agentique

On vient de faire **un tour à la main**. Un agent, c'est simplement **ce tour répété dans une boucle** jusqu'à ce que le modèle n'ait plus besoin d'outil :

```
   ┌──────────────────────────────────────────────┐
   │ 1. Le MODÈLE réfléchit                         │
   │ 2. Il demande un outil (tool_use)  ── non ──►  Réponse finale
   │ 3. VOTRE CODE exécute l'outil                  │
   │ 4. Le résultat est renvoyé au modèle ──► retour en 1
   └──────────────────────────────────────────────┘
        Sécurité : au maximum 8 tours
```

### 4.1 Un deuxième outil : rédiger un message

In [21]:
from prospection import rediger_message, OFFRE

print("Produit présenté :", OFFRE)
print(rediger_message("Boulangerie Teranga (démo)", "boulangerie"))

OUTIL_REDACTION = {
    "name": "rediger_message",
    "description": "Rédige un message WhatsApp d'approche court et personnalisé pour UN commerce.",
    "input_schema": {
        "type": "object",
        "properties": {
            "nom_commerce": {"type": "string", "description": "Nom exact du commerce"},
            "activite": {"type": "string", "description": "Type de commerce"},
            "ton": {"type": "string", "enum": ["chaleureux", "formel", "direct"]},
        },
        "required": ["nom_commerce"],
    },
}

Produit présenté : CaissePro, un logiciel de caisse et de gestion de stock simple, utilisable sur téléphone et compatible Wave et Orange Money
{'commerce': 'Boulangerie Teranga (démo)', 'message': 'Bonjour, je suis Moussa de CaissePro. Nous aidons les boulangeries comme la vôtre à encaisser et suivre leur stock simplement depuis un téléphone, avec paiement par Wave et Orange Money. Seriez-vous intéressé(e) pour que je vous montre comment cela fonctionne ?'}


### 4.2 La boucle complète (environ 40 lignes)
Lisez les commentaires : chaque ligne correspond à une étape du schéma.

In [22]:
OUTILS = [OUTIL_RECHERCHE, OUTIL_REDACTION]
FONCTIONS = {"rechercher_commerces": chercher_commerces, "rediger_message": rediger_message}

SYSTEME = """Tu es un assistant de prospection commerciale au Sénégal.
Trouve des commerces avec rechercher_commerces, puis rédige un message pour chacun
avec rediger_message. Réponds en français, de façon courte et claire."""

def mon_agent(demande, max_tours=8):
    messages = [{"role": "user", "content": demande}]

    for tour in range(1, max_tours + 1):
        # 1. Le modèle réfléchit
        rep = client.messages.create(model=MODELE, max_tokens=2000, system=SYSTEME,
                                     tools=OUTILS, messages=messages)
        messages.append({"role": "assistant", "content": rep.content})

        # Plus d'outil demandé ? C'est la réponse finale.
        if rep.stop_reason != "tool_use":
            print(f"✅ Terminé en {tour} tour(s)")
            return "".join(b.text for b in rep.content if b.type == "text")

        # 2 et 3. Pour chaque outil demandé, notre code l'exécute
        resultats = []
        for bloc in rep.content:
            if bloc.type == "tool_use":
                print(f"🔧 Tour {tour} : {bloc.name}({bloc.input})")
                try:
                    sortie = FONCTIONS[bloc.name](**bloc.input)
                except Exception as e:
                    sortie = {"erreur": str(e)}        # l'erreur devient une information
                resultats.append({"type": "tool_result", "tool_use_id": bloc.id,
                                  "content": json.dumps(sortie, ensure_ascii=False)})

        # 4. On renvoie les résultats au modèle, et on recommence
        messages.append({"role": "user", "content": resultats})

    return "⛔ Arrêt : nombre maximal de tours atteint."

In [23]:
reponse_finale = mon_agent("Trouve 3 boulangeries à Dakar Plateau et prépare un message pour chacune.")
print()
print(reponse_finale)

🔧 Tour 1 : rechercher_commerces({'activite': 'boulangerie', 'zone': 'Dakar Plateau', 'nombre': 3})
🔧 Tour 2 : rediger_message({'nom_commerce': 'Praline', 'activite': 'pâtisserie'})
🔧 Tour 2 : rediger_message({'nom_commerce': 'LA BAGUETTE MAGIQUE', 'activite': 'boulangerie'})
🔧 Tour 2 : rediger_message({'nom_commerce': 'Boulangerie Principale', 'activite': 'boulangerie'})
✅ Terminé en 3 tour(s)

Voici les 3 commerces trouvés et un message pour chacun.

**1. Praline** (pâtisserie, 4,5/5, 69 avis)
- Adresse : 14-18 Bd El Hadji Djily Mbaye
- Tél : +221 33 864 66 74
- Message : « Bonjour, c'est [Votre nom] de CaissePro. Nous proposons un logiciel de caisse et de gestion de stock simple, utilisable sur téléphone et compatible Wave et Orange Money, qui pourrait vous aider au quotidien chez Praline. Seriez-vous intéressé(e) pour que je vous montre comment il fonctionne ? »

**2. La Baguette Magique** (boulangerie, 4/5, 2 avis)
- Adresse : 22 Rue Robert Brun
- Tél : +221 70 313 68 31
- Message 

🎉 **Vous venez de construire un agent.** Tous les frameworks (LangGraph, CrewAI…) font exactement cela, avec des options en plus.

### 4.3 Trois détails importants
1. Le modèle peut demander **plusieurs outils dans une même réponse** : d'où la boucle `for bloc`.
2. Chaque résultat doit porter le `tool_use_id` de sa demande, sinon l'API refuse.
3. Il faut ajouter la réponse de l'assistant à `messages` **avant** les résultats d'outils.

## 5. Quand un outil plante

Que se passe-t-il si un outil lève une erreur ? On ne plante pas : on **renvoie l'erreur au modèle**, qui s'adapte. Testons avec un outil volontairement cassé.

In [24]:
def recherche_capricieuse(activite, zone, nombre=5):
    if "thies" in zone.lower().replace("è", "e"):
        raise ConnectionError("le service Google Maps ne répond pas pour Thiès")
    return chercher_commerces(activite, zone, nombre)

FONCTIONS["rechercher_commerces"] = recherche_capricieuse
print(mon_agent("Trouve des boulangeries à Thiès."))
FONCTIONS["rechercher_commerces"] = chercher_commerces      # on remet la vraie fonction

🔧 Tour 1 : rechercher_commerces({'activite': 'boulangerie', 'zone': 'Thiès'})
🔧 Tour 2 : rechercher_commerces({'activite': 'boulangerie', 'zone': 'Thiès'})
✅ Terminé en 3 tour(s)
Je n'ai pas pu récupérer de boulangeries à Thiès. J'ai essayé deux fois, et le service Google Maps ne répond pas pour cette zone. Je n'ai donc ni liste de commerces ni message à vous proposer.

Voici ce que je peux faire :
- relancer la recherche un peu plus tard ;
- chercher avec une zone plus précise, par exemple « Thiès Nord », « Thiès Ouest » ou « Randoulène » ;
- chercher dans une ville proche, comme Mbour ou Tivaouane.

Dites-moi ce que vous préférez.


👀 L'agent a lu l'erreur et a réagi : il a réessayé, proposé une autre zone, ou expliqué le problème poliment. **C'est la force d'un agent : il s'adapte aux imprévus.**

## 6. ✍️ À vous : trois missions

**Mission 1.** Ajoutez un outil `sauvegarder_prospects(prospects)` qui enregistre une liste de prospects (nom, téléphone, message) dans `donnees/prospects.csv`. Écrivez la fonction **et** sa description JSON Schema, ajoutez-les à `OUTILS` et `FONCTIONS`, puis demandez à l'agent de sauvegarder ses résultats.

**Mission 2.** Testez une demande sans résultat : « Trouve des boulangeries à Kédougou ». Que fait l'agent ?

**Mission 3 (bonus).** Modifiez `mon_agent` pour afficher à la fin le total de tokens consommés (`rep.usage.input_tokens` et `rep.usage.output_tokens`).

In [25]:
# Mission 1 : votre code ici

<details>
<summary><b>💡 Voir la solution</b></summary>

```python
import csv
from pathlib import Path

def sauvegarder_prospects(prospects):
    chemin = Path("../donnees/prospects.csv")
    chemin.parent.mkdir(exist_ok=True)
    with open(chemin, "w", newline="", encoding="utf-8") as f:
        ecrivain = csv.DictWriter(f, fieldnames=["nom", "telephone", "message"])
        ecrivain.writeheader()
        for p in prospects:
            ecrivain.writerow({k: p.get(k, "") for k in ["nom", "telephone", "message"]})
    return {"statut": "ok", "fichier": str(chemin), "lignes": len(prospects)}

OUTIL_SAUVEGARDE = {
    "name": "sauvegarder_prospects",
    "description": "Enregistre la liste finale des prospects (nom, téléphone, message) dans un fichier CSV.",
    "input_schema": {
        "type": "object",
        "properties": {
            "prospects": {
                "type": "array",
                "items": {
                    "type": "object",
                    "properties": {
                        "nom": {"type": "string"},
                        "telephone": {"type": "string"},
                        "message": {"type": "string"},
                    },
                    "required": ["nom", "message"],
                },
            }
        },
        "required": ["prospects"],
    },
}
OUTILS.append(OUTIL_SAUVEGARDE)
FONCTIONS["sauvegarder_prospects"] = sauvegarder_prospects

print(mon_agent("Trouve 3 boulangeries à Dakar Plateau, rédige un message pour chacune, "
                "puis sauvegarde les prospects."))
```
</details>

## 7. 🎁 Bonus : le RAG, donner des documents à l'agent

Pour rédiger un bon message, l'agent doit connaître **votre produit**. Mais on ne peut pas mettre un catalogue de 40 pages dans chaque prompt.

Le **RAG** (*génération augmentée par la recherche*) :
1. **Une fois** : on découpe les documents en passages et on les indexe.
2. **À chaque demande** : on retrouve les quelques passages les plus proches de la question, et on ne donne **que ceux-là** au modèle.

> En production, on indexe avec des **embeddings** (des vecteurs qui représentent le sens). Ici, pour ne pas ajouter de clé API, on utilise **TF-IDF** (scikit-learn), qui compare les mots. Le principe est exactement le même.

In [26]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

FICHE_PRODUIT = """
CaissePro fonctionne sur un simple téléphone Android, sans ordinateur.
Les paiements Wave et Orange Money sont enregistrés automatiquement dans la caisse.
Le suivi du stock alerte le commerçant quand un produit va manquer, par exemple la farine.
Pour les boulangeries, CaissePro gère les commandes de pain à l'avance des restaurants et hôtels.
Pour les pharmacies, CaissePro suit les dates de péremption des médicaments.
L'abonnement mensuel est sans engagement, avec un mois d'essai gratuit.
Une formation d'une heure est offerte sur place, à Dakar et dans les régions.
Les données sont sauvegardées chaque nuit, même en cas de perte du téléphone.
"""

passages = [p.strip() for p in FICHE_PRODUIT.strip().split("\n") if p.strip()]
vectoriseur = TfidfVectorizer()
index = vectoriseur.fit_transform(passages)          # étape 1 : indexation (une fois)

def chercher_dans_catalogue(question, k=2):
    """Étape 2 : renvoie les k passages les plus proches de la question."""
    scores = cosine_similarity(vectoriseur.transform([question]), index)[0]
    meilleurs = scores.argsort()[::-1][:k]
    return [passages[i] for i in meilleurs]

print(chercher_dans_catalogue("argument pour une boulangerie"))
print(chercher_dans_catalogue("pharmacie médicaments"))

["Une formation d'une heure est offerte sur place, à Dakar et dans les régions.", 'Pour les pharmacies, CaissePro suit les dates de péremption des médicaments.']
['Pour les pharmacies, CaissePro suit les dates de péremption des médicaments.', 'Les données sont sauvegardées chaque nuit, même en cas de perte du téléphone.']


Le RAG devient simplement **un outil de plus** pour notre agent :

In [14]:
OUTIL_CATALOGUE = {
    "name": "chercher_dans_catalogue",
    "description": ("Cherche dans la fiche produit les arguments adaptés à un type de commerce. "
                    "À utiliser AVANT de rédiger un message, pour le personnaliser."),
    "input_schema": {"type": "object",
                     "properties": {"question": {"type": "string"}},
                     "required": ["question"]},
}
OUTILS.append(OUTIL_CATALOGUE)
FONCTIONS["chercher_dans_catalogue"] = chercher_dans_catalogue

print(mon_agent("Trouve 2 pharmacies aux Almadies et prépare un message personnalisé "
                "avec un argument tiré du catalogue pour chacune."))

Unexpected exception formatting exception. Falling back to standard exception


Traceback (most recent call last):
  File "c:\Users\Ultra Tech\anaconda3\Lib\site-packages\IPython\core\interactiveshell.py", line 3508, in run_code
    exec(code_obj, self.user_global_ns, self.user_ns)
  File "C:\Users\Ultra Tech\AppData\Local\Temp\ipykernel_16200\2199085164.py", line 12, in <module>
    print(mon_agent("Trouve 2 pharmacies aux Almadies et prépare un message personnalisé "
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\Ultra Tech\AppData\Local\Temp\ipykernel_16200\1349899430.py", line 13, in mon_agent
    rep = client.messages.create(model=MODELE, max_tokens=2000, system=SYSTEME,
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\Ultra Tech\anaconda3\Lib\site-packages\anthropic\_utils\_utils.py", line 282, in wrapper
    return func(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\Ultra Tech\anaconda3\Lib\site-packages\anthropic\resources\messag

## 8. 🎁 Bonus : MCP, un outil écrit une fois, utilisable partout

Notre outil de recherche vit dans notre code. **MCP** (*Model Context Protocol*) est un standard qui permet de l'exposer une fois pour toutes : Claude Desktop, un éditeur de code ou un autre agent peuvent alors s'y brancher.

La cellule suivante **écrit** un petit serveur MCP dans le fichier `serveur_mcp_prospection.py`.

In [27]:
%%writefile serveur_mcp_prospection.py
"""Serveur MCP qui expose la recherche de commerces."""
from mcp.server.fastmcp import FastMCP
from prospection import chercher_commerces

serveur = FastMCP("prospection")

@serveur.tool()
def rechercher_commerces(activite: str, zone: str, nombre: int = 5) -> list[dict]:
    """Recherche des commerces sur Google Maps au Sénégal par activité et zone."""
    return chercher_commerces(activite, zone, nombre)

if __name__ == "__main__":
    serveur.run()          # communique avec l'assistant par l'entrée/sortie standard

Overwriting serveur_mcp_prospection.py


**Pour le brancher sur Claude Desktop :** menu *Paramètres → Développeur → Modifier la configuration*, puis ajoutez (en adaptant les chemins) :

```json
{
  "mcpServers": {
    "prospection": {
      "command": "python",
      "args": ["C:/chemin/vers/atelier-agents-ia/notebooks/serveur_mcp_prospection.py"]
    }
  }
}
```
Redémarrez Claude Desktop et demandez : *« Trouve 3 pharmacies aux Almadies »*.

> **MCP relie un agent à des outils.** Au notebook 04, on verra **A2A**, qui relie un agent à **un autre agent**.

## 🎯 Récapitulatif

| Notion | Ce qu'il faut retenir |
|---|---|
| Outil | Une fonction Python + une description JSON Schema |
| Function calling | Le modèle **demande** (`tool_use`), votre code **exécute** |
| Boucle agentique | Répéter : modèle → outil → résultat, jusqu'à la réponse finale |
| Erreur d'outil | On la renvoie au modèle, il s'adapte |
| RAG | Retrouver les bons passages et ne donner qu'eux au modèle |
| MCP | Standard pour partager un outil entre plusieurs assistants |

➡️ **Prochaine étape : notebook 03.** Notre agent peut agir dans le monde réel… il faut donc le **protéger**.